[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/23_cross_attention.ipynb)

# 🟡 Medium: Cross-Attention

*Attention & Transformers*
Implement **multi-head cross-attention**: queries come from one sequence, keys
and values from another.

$$\text{CrossAttn}(x_q, x_{kv}) = \text{softmax}\!\left(
\frac{(x_q W^Q)(x_{kv} W^K)^\top}{\sqrt{d_k}}\right)(x_{kv} W^V)\,W^O$$

### Signature
```python
class MultiHeadCrossAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs): ...
    def __call__(self, x_q, x_kv): ...
```

### Requirements
- Use `nnx.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `self.d_k = d_model // num_heads`
- `x_q` is `(B, S_q, d_model)`, `x_kv` is `(B, S_kv, d_model)`
- Output is `(B, S_q, d_model)` — the **query** length

### Self-attention vs cross-attention
Structurally they are the same computation; the difference is entirely in what
gets projected:

| | Q from | K, V from |
|---|---|---|
| self-attention | `x` | `x` |
| cross-attention | `x_q` | `x_kv` |

So `cross(x, x)` is exactly self-attention. Everything else — the heads, the
scaling, the softmax axis — is unchanged.

### Where it shows up
- **Encoder-decoder** transformers: the decoder queries the encoder's output,
  which is how translation conditions on the source sentence.
- **Diffusion models**: image latents query text embeddings — this is the
  single point where the prompt enters a U-Net.
- **Perceiver / Flamingo**: a small set of learned latent queries attends over a
  large input, decoupling compute from input size.

### Why the output follows the query
Each query position produces exactly one output row, no matter how many keys it
attended over. That is what lets cross-attention consume a context of a
completely different length — a 1000-token document can condition a 10-token
generation, and the cost is $O(S_q S_{kv})$ rather than anything quadratic in
the larger one alone.

### The trap
Reading a single sequence length off `x_q` and reusing it for `x_kv` passes
every equal-length test and then fails the moment the two differ. Read both.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class MultiHeadCrossAttention(nnx.Module):
    """Queries from x_q attend over keys/values from x_kv."""

    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, x_q, x_kv):
        """(B, S_q, d_model), (B, S_kv, d_model) -> (B, S_q, d_model)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

ca = MultiHeadCrossAttention(32, 4, rngs=nnx.Rngs(params=0))

dec = jax.random.normal(jax.random.key(1), (1, 4, 32))    # 4 decoder tokens
enc = jax.random.normal(jax.random.key(2), (1, 20, 32))   # 20 encoder tokens

print("decoder:", dec.shape, " encoder:", enc.shape)
print("output :", ca(dec, enc).shape, "(follows the query length)")
print("self-attention as a special case:", ca(dec, dec).shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("cross_attention")

# hint("cross_attention")      # stuck? nudge without the answer
# solution("cross_attention")  # spoiler: the reference implementation